# **ALGORITMOS DE OPTIMIZACIÓN**

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
# — Datos sintéticos: make_moons con ruido
np.random.seed(1)
tf.random.set_seed(1)
X, y = make_moons(n_samples=1000, noise=0.25, random_state=1)
X = StandardScaler().fit_transform(X)

# Split train/validation
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=1
)

# — Modelo base de clasificación binaria
def build_model():
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(64, activation='relu', input_shape=(2,)),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])
    return model

# — Optimizadores a comparar
opts = {
    'SGD (lr=0.1)':   tf.keras.optimizers.SGD(learning_rate=0.1),
    'Adam (lr=0.01)': tf.keras.optimizers.Adam(learning_rate=0.01)
}

histories = {}
for name, opt in opts.items():
    m = build_model()
    m.compile(optimizer=opt,
              loss='binary_crossentropy',
              metrics=['accuracy'])
    h = m.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=60,
        batch_size=32,
        verbose=0
    )
    histories[name] = h

# — Gráficas comparativas
plt.figure(figsize=(12,5))

# 1) Accuracy
plt.subplot(1,2,1)
for name, h in histories.items():
    plt.plot(h.history['accuracy'],      label=f'{name} – train')
    plt.plot(h.history['val_accuracy'], '--', label=f'{name} – val')
plt.title('Precisión: entrenamiento vs validación')
plt.xlabel('Época')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# 2) Loss
plt.subplot(1,2,2)
for name, h in histories.items():
    plt.plot(h.history['loss'],      label=f'{name} – train')
    plt.plot(h.history['val_loss'], '--', label=f'{name} – val')
plt.title('Pérdida (binary_crossentropy)')
plt.xlabel('Época')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()